# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

For my lane, **one row = one content page (uniquely defined by `client_hash_id` and `content_hash_id`) aggregated over a single calendar month.**

For this contract, I am using the mid-panel month of **March 2026** (from `2026-03-01` to `2026-03-31`). The features are built from performance signals captured entirely within this March window. The target label represents whether the page declined over the *next* month (April 2026).

In [1]:
import duckdb
import os

# Retrieve Hugging Face read token from local environment config
HF_TOKEN = os.environ.get('HF_TOKEN', '')
if not HF_TOKEN:
    # Read from local git-ignored env file if running locally
    if os.path.exists('../../.env'):
        with open('../../.env') as f:
            for line in f:
                if line.startswith('HF_TOKEN='):
                    HF_TOKEN = line.split('=', 1)[1].strip()

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
fact_daily_march = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"

# Verify total row count and date bounds for the month
print("Verifying March 2026 daily Performance data:")
summary = con.sql(f"""
    SELECT 
        COUNT(*) as total_rows,
        MIN(report_date) as start_date,
        MAX(report_date) as end_date
    FROM {fact_daily_march}
""").df()
print(summary.to_string(index=False))

Verifying March 2026 daily Performance data:


 total_rows start_date   end_date
    9841378 2026-03-01 2026-03-31


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

I classify the fields into the following categories:

*   **Features (Knowable before the decision moment):**
    *   `march_impressions`: Sum of organic GSC impressions in March 2026.
    *   `march_clicks`: Sum of organic GSC clicks in March 2026.
    *   `march_avg_position`: Mean GSC average position in March 2026.
    *   `march_pageviews`: Sum of GA4 pageviews in March 2026.
    *   `word_count`: Length of the page's content.
    *   `ctr`: Calculated click-through rate (`march_clicks` / `march_impressions`).
*   **Label / Proxy (What I am predicting):**
    *   `is_declining`: Declining flag (1 = impressions declined in April 2026 by more than 20% compared to March 2026, 0 = stable/recovering).
*   **Context (For splits, joins, and metadata):**
    *   `client_hash_id`: Pseudonymized ID of the client.
    *   `content_hash_id`: Pseudonymized ID of the content page.
*   **Excluded (With a clear why):**
    *   `trend_direction` and `trend_pct` from the main snapshot: Excluded because they contain the target metrics from the future prediction window, creating major target leakage.

In [2]:
# Quick verification that the daily grain (client_hash_id x content_hash_id x report_date) contains no duplicates
print("Checking daily performance table grain for March 2026:")
grain_check = con.sql(f"""
    SELECT client_hash_id, content_hash_id, report_date, COUNT(*) as c
    FROM {fact_daily_march}
    GROUP BY 1, 2, 3
    HAVING c > 1
    LIMIT 5
""").df()

if grain_check.empty:
    print("  Success: Daily grain is clean (0 duplicates found).")
else:
    print("  Warning: Found duplicate grain rows!")
    print(grain_check)

Checking daily performance table grain for March 2026:


  Success: Daily grain is clean (0 duplicates found).


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

I run verification checks to verify: 
1. Total aggregated page counts for the month.
2. Availability of data sources (GSC vs GA4 flags).
3. Visualizing how many rows survive when requiring both systems.

In [3]:
# Query showing GSC and GA4 data availability counts
print("Data availability check for March 2026:")
availability = con.sql(f"""
    SELECT 
        COUNT(*) as total_daily_rows,
        SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) as gsc_available_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) as ga4_available_rows,
        SUM(CASE WHEN gsc_data_available IS TRUE AND ga4_data_available IS TRUE THEN 1 ELSE 0 END) as both_available_rows
    FROM {fact_daily_march}
""").df()
print(availability.to_string(index=False))

Data availability check for March 2026:


 total_daily_rows  gsc_available_rows  ga4_available_rows  both_available_rows
          9841378           3611061.0            413966.0             364347.0


## 4. Five features + Leakage Experiment

*Build a feature frame, specify the 'available when?' logic, and run the leakage test.*

In [4]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

fact_april = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-04/*.parquet')"
dim_content = f"read_parquet('{REL}/dim_content.parquet')"

# Build monthly feature frame from March and label from April
print("Querying and aggregating features (March) and label (April)... ")
data_df = con.sql(f"""
    WITH march_agg AS (
        SELECT 
            client_hash_id, 
            content_hash_id,
            SUM(gsc_impressions) as march_impressions,
            SUM(gsc_clicks) as march_clicks,
            AVG(gsc_avg_position) as march_avg_position,
            SUM(ga4_pageviews) as march_pageviews
        FROM {fact_daily_march}
        WHERE gsc_data_available IS TRUE
        GROUP BY 1, 2
        HAVING march_impressions >= 50
    ),
    april_agg AS (
        SELECT 
            client_hash_id, 
            content_hash_id,
            SUM(gsc_impressions) as april_impressions
        FROM {fact_april}
        WHERE gsc_data_available IS TRUE
        GROUP BY 1, 2
    ),
    metadata AS (
        SELECT DISTINCT client_hash_id, content_hash_id, word_count
        FROM {dim_content}
    )
    SELECT 
        m.march_impressions,
        m.march_clicks,
        m.march_avg_position,
        m.march_pageviews,
        COALESCE(a.april_impressions, 0) as april_impressions, -- Leakage trap feature
        COALESCE(wc.word_count, 0) as word_count
    FROM march_agg m
    LEFT JOIN april_agg a ON m.client_hash_id = a.client_hash_id AND m.content_hash_id = a.content_hash_id
    LEFT JOIN metadata wc ON m.client_hash_id = wc.client_hash_id AND m.content_hash_id = wc.content_hash_id
""").df()

# Construct honest features
data_df['ctr'] = data_df['march_clicks'] / data_df['march_impressions']
data_df['is_declining'] = (data_df['april_impressions'] < 0.8 * data_df['march_impressions']).astype(int)
data_df = data_df.dropna(subset=['march_impressions', 'march_clicks', 'march_avg_position', 'word_count', 'ctr'])

print(f"Aggregated dataset shape: {data_df.shape[0]:,} pages")
print(f"Class balance (decline rate): {data_df['is_declining'].mean() * 100:.1f}%")

# Split train / test
y = data_df['is_declining']
X_honest = data_df[['march_impressions', 'march_clicks', 'march_avg_position', 'word_count', 'ctr']]

X_tr_h, X_te_h, y_tr, y_te = train_test_split(X_honest, y, test_size=0.25, random_state=42)

# 1. Train Honest Model
clf_h = RandomForestClassifier(n_estimators=50, max_depth=8, random_state=42, n_jobs=-1).fit(X_tr_h, y_tr)
score_h = roc_auc_score(y_te, clf_h.predict_proba(X_te_h)[:, 1])
print(f"Honest Model ROC-AUC:  {score_h:.4f}")

# 2. Train Leakage Model (including future april_impressions)
X_leak = data_df[['march_impressions', 'march_clicks', 'march_avg_position', 'word_count', 'ctr', 'april_impressions']]
X_tr_l, X_te_l, _, _ = train_test_split(X_leak, y, test_size=0.25, random_state=42)

clf_l = RandomForestClassifier(n_estimators=50, max_depth=8, random_state=42, n_jobs=-1).fit(X_tr_l, y_tr)
score_l = roc_auc_score(y_te, clf_l.predict_proba(X_te_l)[:, 1])
print(f"Leakage Model ROC-AUC: {score_l:.4f}")

Querying and aggregating features (March) and label (April)... 


Aggregated dataset shape: 116,114 pages
Class balance (decline rate): 51.8%


Honest Model ROC-AUC:  0.6758


Leakage Model ROC-AUC: 0.9799


## 5. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**GA4 page tracking data is highly sparse and restricted.** Looking at March 2026 performance, while GSC has 3.6 million active tracking rows, only ~400,000 daily rows have GA4 data enabled, and only 364,000 rows contain both metrics. This means that if I require GA4 metrics (like pageviews or scroll events) for feature engineering, I am forced to drop ~90% of the active search pages in the dataset. Models relying heavily on engagement data will suffer from severely reduced sample sizes, which is a major limitation of this warehouse release.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.